In [1]:
import os
import numpy as np
import tensorflow as tf

In [ ]:
class DeepQNetwork(object):
    def __init__(self, lr, n_actions, name, fcl_dims=256, input_dims=(210, 160, 4), chkpt_dir='tmp/dqn'):
        self.lr = lr        
        self.n_actions = n_actions
        self.name = name
        self.fcl_dims = fcl_dims
        self.input_dims = input_dims
        self.chkpt_dir = chkpt_dir
        self.sess = tf.Session()
        self.build_network()
        self.sess.run(tf.global_variables_initializer())
        self.saver = tf.train.Saver()
        self.checkout_file = os.path.join(self.chkpt_dir, 'deepqnet.ckpt')
        self.params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=self.name)
    def build_network(self):
        with tf.variable_scope(self.name):
            self.input = tf.placeholder(tf.float32, shape=[None, *self.input_dims], name='input')
            self.actions = tf.placeholder(tf.int32, shape=[None], name='actions')
            self.target_Q = tf.placeholder(tf.float32, shape=[None], name='target')
            flat = tf.layers.flatten(self.input)
            dense1 = tf.layers.dense(flat, self.fcl_dims, activation=tf.nn.relu)
            dense2 = tf.layers.dense(dense1, self.fcl_dims, activation=tf.nn.relu)
            self.Q_values = tf.layers.dense(dense2, self.n_actions)
            action_indices = tf.stack([tf.range(tf.shape(self.actions)[0]), self.actions], axis=1)
            self.Q_action = tf.gather_nd(params=self.Q_values, indices=action_indices)
            self.loss = tf.reduce_mean(tf.square(self.target_Q - self.Q_action))
            self.optimizer = tf.train.AdamOptimizer(learning_rate=self.lr).minimize(self.loss)